In [2]:
%pip install beautifulsoup4 fuzzywuzzy python-Levenshtein

Note: you may need to restart the kernel to use updated packages.


In [3]:
from bs4 import BeautifulSoup
import csv
import re

import pandas as pd
import pycountry



In [4]:
# Read the HTML file
with open('../Raw/IIA Content Mapping _ International Investment Agreements Navigator _ UNCTAD Investment Policy Hub.html', 'r', encoding='utf-8') as f:
    html = f.read()

soup = BeautifulSoup(html, 'html.parser')

# Find all rows in the table body
rows = soup.select('table.table tbody tr')

results = []

for row in rows:
    cells = row.find_all(['td', 'th'])
    
    if len(cells) < 2:
        continue
    
    # Extract number
    number = cells[0].get_text(strip=True)
    
    # Extract short title and URL
    short_title_cell = row.find('td', {'data-index': '2'})
    if not short_title_cell:
        continue
    
    short_title_link = short_title_cell.find('a')
    short_title = short_title_link.get_text(strip=True) if short_title_link else ''
    treaty_url = short_title_link.get('href', '') if short_title_link else ''
    if treaty_url:
        treaty_url = 'https://investmentpolicy.unctad.org' + treaty_url
    
    # Skip empty rows
    if not short_title:
        continue
    
    # Extract type
    type_cell = row.find('td', {'data-index': '3'})
    treaty_type = type_cell.get_text(strip=True) if type_cell else ''
    
    # Extract status
    status_cell = row.find('td', {'data-index': '4'})
    status = status_cell.get_text(strip=True) if status_cell else ''
    
    # Extract parties
    parties_cell = row.find('td', {'data-index': '5'})
    if parties_cell:
        party_links = parties_cell.find_all('a')
        parties = ' | '.join([p.get_text(strip=True) for p in party_links])
    else:
        parties = ''
    
    # Extract signature date
    sig_date_cell = row.find('td', {'data-index': '6'})
    signature_date = sig_date_cell.get_text(strip=True) if sig_date_cell else ''
    
    # Extract entry into force date
    eif_date_cell = row.find('td', {'data-index': '7'})
    entry_into_force_date = eif_date_cell.get_text(strip=True) if eif_date_cell else ''
    
    # Extract termination date
    term_date_cell = row.find('td', {'data-index': '8'})
    termination_date = term_date_cell.get_text(strip=True) if term_date_cell else ''
    
    # Extract text links
    text_cell = row.find('td', {'data-index': '9'})
    if text_cell:
        text_links = text_cell.find_all('a')
        text_urls = ' | '.join([
            f"{a.get_text(strip=True)}: https://investmentpolicy.unctad.org{a.get('href', '')}"
            for a in text_links
        ])
    else:
        text_urls = ''
    
    results.append({
        'number': number,
        'short_title': short_title,
        'treaty_url': treaty_url,
        'type': treaty_type,
        'status': status,
        'parties': parties,
        'signature_date': signature_date,
        'entry_into_force_date': entry_into_force_date,
        'termination_date': termination_date,
        'text_urls': text_urls
    })

In [5]:
results_df = pd.DataFrame(results)

results_df["signature_date"] = pd.to_datetime(results_df["signature_date"], errors='coerce')
results_df["entry_into_force_date"] = pd.to_datetime(results_df["entry_into_force_date"], errors='coerce')
results_df["termination_date"] = pd.to_datetime(results_df["termination_date"], errors='coerce')

results_df.head()

/var/folders/_x/74827dzs033cwdj2j4gch59r0000gn/T/ipykernel_5781/3995723907.py:3: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  results_df["signature_date"] = pd.to_datetime(results_df["signature_date"], errors='coerce')
/var/folders/_x/74827dzs033cwdj2j4gch59r0000gn/T/ipykernel_5781/3995723907.py:4: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  results_df["entry_into_force_date"] = pd.to_datetime(results_df["entry_into_force_date"], errors='coerce')


,number,short_title,treaty_url,type,status,parties,signature_date,entry_into_force_date,termination_date,text_urls
0,1,"Venezuela, Bolivarian Republic of - Viet Nam B...",https://investmentpolicy.unctad.org/internatio...,BITs,In force,"Venezuela, Bolivarian Republic of | Viet Nam",2008-11-20,2009-06-17,NaT,es: https://investmentpolicy.unctad.org/intern...
1,2,USMCA (2018),https://investmentpolicy.unctad.org/internatio...,TIPs,In force,Canada | Mexico | United States of America,2018-11-30,2020-07-01,NaT,en: https://investmentpolicy.unctad.org/intern...
2,3,Uruguay - Viet Nam BIT (2009),https://investmentpolicy.unctad.org/internatio...,BITs,In force,Uruguay | Viet Nam,2009-05-12,2012-09-09,NaT,es: https://investmentpolicy.unctad.org/intern...
3,4,"Uruguay - Venezuela, Bolivarian Republic of BI...",https://investmentpolicy.unctad.org/internatio...,BITs,In force,"Uruguay | Venezuela, Bolivarian Republic of",1997-05-20,2002-01-18,NaT,es: https://investmentpolicy.unctad.org/intern...
4,5,United States - Viet Nam Trade Relations Agree...,https://investmentpolicy.unctad.org/internatio...,TIPs,In force,United States of America | Viet Nam,2000-07-13,2000-07-13,NaT,en: https://investmentpolicy.unctad.org/intern...


In [6]:
# Extract number of parties.
results_df['num_parties'] = results_df['parties'].apply(lambda x: len(x.split(' | ')) if x else 0)
results_df["num_parties"].describe()

count    2808.000000
mean        2.016738
std         0.400487
min         1.000000
25%         2.000000
50%         2.000000
75%         2.000000
max        14.000000
Name: num_parties, dtype: float64

In [7]:
# Only use the treaties with 2 parties, i.e., BITs.
bit_df = results_df[results_df['num_parties'] == 2].copy()
# Make sure both are not empty strings
bit_df = bit_df[(bit_df['parties'] != '') & (bit_df['parties'] != ' | ')]
bit_df.shape

(2776, 11)

In [8]:
# Map the names to their iso codes, though it wont perfectly
# match so we need to use fuzzy matchin
country_to_iso = {country.name: country.alpha_3 for country in pycountry.countries}

In [9]:
# Fuzzy match
from fuzzywuzzy import fuzz

def fuzzy_score(str1, str2):
    """Return fuzzy match score (0-100) between two strings."""
    return fuzz.token_set_ratio(str1, str2)

In [10]:
# Normalise the country names to lowercase 
# and only allow letters and spaces to improve matching
def normalize_country_name(name):
    name = name.lower()
    name = re.sub(r'[^a-z\s]', '', name)
    return name.strip()

norm_country_to_iso = {normalize_country_name(name): code for name, code in country_to_iso.items()}
bit_df["norm_party1"] = bit_df['parties'].apply(lambda x: normalize_country_name(x.split(' | ')[0]) if x else '')
bit_df["norm_party2"] = bit_df['parties'].apply(lambda x: normalize_country_name(x.split(' | ')[1]) if x and len(x.split(' | ')) > 1 else '')

bit_df[["parties", "norm_party1", "norm_party2"]].head()

,parties,norm_party1,norm_party2
0,"Venezuela, Bolivarian Republic of | Viet Nam",venezuela bolivarian republic of,viet nam
2,Uruguay | Viet Nam,uruguay,viet nam
3,"Uruguay | Venezuela, Bolivarian Republic of",uruguay,venezuela bolivarian republic of
4,United States of America | Viet Nam,united states of america,viet nam
5,United States of America | Uzbekistan,united states of america,uzbekistan


In [11]:
# Now map them, first see if they already exist in normalised country map
# otherwise use fuzzy matching to find the best match.
from tqdm import tqdm

threshold = 80  # Adjust as needed

def map_to_iso(norm_name):
    if norm_name in norm_country_to_iso:
        return norm_country_to_iso[norm_name]
    
    best_match = None
    best_score = 0
    
    for country_name in norm_country_to_iso.keys():
        score = fuzzy_score(norm_name, country_name)
        if score > best_score:
            best_score = score
            best_match = country_name
            
    if best_score >= threshold:
        return norm_country_to_iso[best_match]
    
    return None

tqdm.pandas()
bit_df['iso_party1'] = bit_df['norm_party1'].progress_apply(map_to_iso)
tqdm.pandas()
bit_df['iso_party2'] = bit_df['norm_party2'].progress_apply(map_to_iso)

# Print how many were successfully mapped
print(f"Party 1 mapped: {bit_df['iso_party1'].notnull().sum()} / {len(bit_df)}")
print(f"Party 2 mapped: {bit_df['iso_party2'].notnull().sum()} / {len(bit_df)}")

100%|██████████| 2776/2776 [00:00<00:00, 7834.46it/s]

Party 1 mapped: 2671 / 2776
Party 2 mapped: 2762 / 2776


In [12]:
bit_df[["parties", "norm_party1", "norm_party2", "iso_party1", "iso_party2"]].head()

,parties,norm_party1,norm_party2,iso_party1,iso_party2
0,"Venezuela, Bolivarian Republic of | Viet Nam",venezuela bolivarian republic of,viet nam,VEN,VNM
2,Uruguay | Viet Nam,uruguay,viet nam,URY,VNM
3,"Uruguay | Venezuela, Bolivarian Republic of",uruguay,venezuela bolivarian republic of,URY,VEN
4,United States of America | Viet Nam,united states of america,viet nam,USA,VNM
5,United States of America | Uzbekistan,united states of america,uzbekistan,USA,UZB


In [22]:
# Create bilateral pairs (both directions)
bit_forward = bit_df[['iso_party1', 'iso_party2', 'short_title', 'signature_date', 'entry_into_force_date']].copy()
bit_backward = bit_df[['iso_party2', 'iso_party1', 'short_title', 'signature_date', 'entry_into_force_date']].copy()
bit_backward.columns = ['iso_party1', 'iso_party2', 'short_title', 'signature_date', 'entry_into_force_date']

# Combine both directions
bit_bilateral = pd.concat([bit_forward, bit_backward], ignore_index=True)

# Sort by entry_into_force_date descending and drop duplicates (keep latest)
bit_bilateral = bit_bilateral.sort_values('entry_into_force_date', ascending=False, na_position='last')
bit_bilateral = bit_bilateral.drop_duplicates(subset=['iso_party1', 'iso_party2'], keep='first')

# Rename for clarity
exp_df = bit_bilateral.rename(columns={
    'short_title': 'bit_name',
    'signature_date': 'bit_signature_date',
    'entry_into_force_date': 'bit_entry_into_force_date'
})

exp_df["bit_year"] = exp_df['bit_entry_into_force_date'].dt.year

# Remove NaN by iso codes or year
exp_df = exp_df.dropna(subset=['iso_party1', 'iso_party2', 'bit_year'])

# Rename iso_party1 to iso3_i, and iso_party2 to iso3_j for easier merging later
exp_df = exp_df.rename(columns={
    'iso_party1': 'iso3_i',
    'iso_party2': 'iso3_j'
})

print(f"Total bilateral pairs: {len(exp_df)}")
exp_df.head(10)

Total bilateral pairs: 4387


,iso3_i,iso3_j,bit_name,bit_signature_date,bit_entry_into_force_date,bit_year
3829,TUR,CHN,"Hong Kong, China SAR - Türkiye BIT (2023)",2023-10-31,2026-02-04,2026.0
1053,CHN,TUR,"Hong Kong, China SAR - Türkiye BIT (2023)",2023-10-31,2026-02-04,2026.0
2191,BRA,IND,Brazil - India BIT (2020),2020-01-25,2025-12-21,2025.0
4967,IND,BRA,Brazil - India BIT (2020),2020-01-25,2025-12-21,2025.0
3114,ARE,NZL,New Zealand - United Arab Emirates BIT (2025),2025-01-14,2025-11-14,2025.0
338,NZL,ARE,New Zealand - United Arab Emirates BIT (2025),2025-01-14,2025-11-14,2025.0
2557,AUS,ARE,Australia - United Arab Emirates BIT (2024),2024-11-06,2025-10-01,2025.0
5333,ARE,AUS,Australia - United Arab Emirates BIT (2024),2024-11-06,2025-10-01,2025.0
3103,SGP,NGA,Nigeria - Singapore BIT (2016),2016-11-04,2025-08-22,2025.0
327,NGA,SGP,Nigeria - Singapore BIT (2016),2016-11-04,2025-08-22,2025.0


In [23]:
# Make sure we only have one entry per pair of countries, and that we keep the most recent BIT if there are multiple.
exp_df["key"] = exp_df.apply(lambda row: row["iso3_i"] + "_" + row["iso3_j"], axis=1)
exp_df["key"].value_counts().sort_values(ascending=False).head()

key
TUR_CHN    1
CHN_TUR    1
BRA_IND    1
IND_BRA    1
ARE_NZL    1
Name: count, dtype: int64

In [25]:
# Export
cols = ["iso3_i", "iso3_j", "bit_year"]
exp_df[cols].to_csv('../Clean/bilateral_investment_treaties.csv', index=False)